# Generate and Join Datasets
This notebook runs the synthetic data generator and creates a unified dataset by joining all tables.

## Step 1: Generate Synthetic Data
Run the data generation script to create all CSV files.

In [ ]:
import subprocess
import os

# Run the synthetic data generator
# Parameters: adjust as needed
cmd = [
    'python',
    'data/generate_synthetic_data.py',
    '--n_customers', '500',
    '--n_products', '200',
    '--n_stores', '20',
    '--n_transactions', '5000',
    '--start_signup', '2018-01-01',
    '--end_signup', '2023-12-31',
    '--start_date', '2023-01-01',
    '--end_date', '2023-12-31',
    '--out_dir', 'data/output',
    '--seed', '42'
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('Errors:', result.stderr)

## Step 2: Load Individual Datasets

In [ ]:
import pandas as pd
import os

# Set data directory
data_dir = 'data/output'

# Load all datasets
customers_df = pd.read_csv(os.path.join(data_dir, 'customers.csv'))
products_df = pd.read_csv(os.path.join(data_dir, 'products.csv'))
stores_df = pd.read_csv(os.path.join(data_dir, 'stores.csv'))
transactions_df = pd.read_csv(os.path.join(data_dir, 'transactions.csv'))
transactions_enriched_df = pd.read_csv(os.path.join(data_dir, 'transactions_enriched.csv'))

print(f"Customers shape: {customers_df.shape}")
print(f"Products shape: {products_df.shape}")
print(f"Stores shape: {stores_df.shape}")
print(f"Transactions shape: {transactions_df.shape}")
print(f"Transactions Enriched shape: {transactions_enriched_df.shape}")

## Step 3: Display Sample Data

In [ ]:
print("\n=== CUSTOMERS (first 5 rows) ===")
print(customers_df.head())
print(f"\nColumns: {customers_df.columns.tolist()}")

print("\n=== PRODUCTS (first 5 rows) ===")
print(products_df.head())
print(f"\nColumns: {products_df.columns.tolist()}")

print("\n=== STORES (first 5 rows) ===")
print(stores_df.head())
print(f"\nColumns: {stores_df.columns.tolist()}")

print("\n=== TRANSACTIONS (first 5 rows) ===")
print(transactions_df.head())
print(f"\nColumns: {transactions_df.columns.tolist()}")

## Step 4: Create Unified Dataset
Join all dimensions with transactions to create a single comprehensive dataset.

In [ ]:
# The transactions_enriched dataset already has most joins,
# but let's verify and create the fully unified dataset explicitly

# Start with transactions
unified_df = transactions_df.copy()

# Join with customers (left join to preserve all transactions)
unified_df = unified_df.merge(
    customers_df,
    on='customer_id',
    how='left'
)

# Join with products
unified_df = unified_df.merge(
    products_df,
    on='product_id',
    how='left'
)

# Join with stores
unified_df = unified_df.merge(
    stores_df,
    on='store_id',
    how='left'
)

print(f"Unified dataset shape: {unified_df.shape}")
print(f"\nTotal columns: {len(unified_df.columns)}")
print(f"\nColumn names:")
print(unified_df.columns.tolist())

## Step 5: Inspect Unified Dataset

In [ ]:
print("\n=== UNIFIED DATASET (first 10 rows) ===")
print(unified_df.head(10))

print("\n=== DATA TYPES ===")
print(unified_df.dtypes)

print("\n=== MISSING VALUES ===")
print(unified_df.isnull().sum())

print("\n=== BASIC STATISTICS ===")
print(unified_df.describe())

## Step 6: Save Unified Dataset

In [ ]:
# Save the unified dataset
output_path = os.path.join(data_dir, 'unified_dataset.csv')
unified_df.to_csv(output_path, index=False)
print(f"Unified dataset saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

## Step 7: Basic Analysis on Unified Dataset

In [ ]:
print("\n=== TRANSACTION SUMMARY ===")
print(f"Total transactions: {len(unified_df)}")
print(f"Total revenue: ${unified_df['total_amount'].sum():,.2f}")
print(f"Average transaction value: ${unified_df['total_amount'].mean():,.2f}")
print(f"Date range: {unified_df['transaction_date'].min()} to {unified_df['transaction_date'].max()}")

print("\n=== TOP 10 PRODUCTS BY REVENUE ===")
top_products = unified_df.groupby('product_name').agg({
    'total_amount': 'sum',
    'quantity': 'sum',
    'transaction_id': 'count'
}).rename(columns={'total_amount': 'revenue', 'quantity': 'units_sold', 'transaction_id': 'num_transactions'}).sort_values('revenue', ascending=False).head(10)
print(top_products)

print("\n=== TOP 10 COUNTRIES BY REVENUE ===")
top_countries = unified_df.groupby('country')['total_amount'].sum().sort_values(ascending=False).head(10)
print(top_countries)

print("\n=== REVENUE BY CATEGORY ===")
category_revenue = unified_df.groupby('category').agg({
    'total_amount': 'sum',
    'transaction_id': 'count'
}).rename(columns={'total_amount': 'revenue', 'transaction_id': 'num_transactions'}).sort_values('revenue', ascending=False)
print(category_revenue)

print("\n=== REVENUE BY STORE SIZE ===")
store_size_revenue = unified_df.groupby('store_size')['total_amount'].sum().sort_values(ascending=False)
print(store_size_revenue)

print("\n=== PAYMENT METHOD DISTRIBUTION ===")
payment_dist = unified_df['payment_method'].value_counts()
print(payment_dist)

print("\n=== PROMOTION IMPACT ===")
promo_stats = unified_df.groupby('promotion').agg({
    'total_amount': ['sum', 'mean', 'count']
}).round(2)
print(promo_stats)